# День 2 · оценка: четыре ретривера на golden-наборе

Как и в `index.ipynb`, код ниже — последовательность ячеек, а не функция: после запуска в
переменных `golden`, `embedder`, `store` останутся реальные объекты, которые можно опросить
отдельной ячейкой. `COLLECTION` внизу должен совпадать с тем, что ты построил в `index.ipynb`.

`evaluate()` и `is_hit()` живут в `common.py`, не здесь: их же импортирует день 3 (`eval_pg.ipynb`),
чтобы сравнение Chroma и pgvector было по одной и той же функции, а не по двум похожим копиям.

In [ ]:
import time

import labkit                          # noqa: F401  подключает .env
from common import evaluate, golden_for_rows, load_config, load_golden, load_rows

# --- НАСТРОЙКИ ---
COLLECTION = "chunks_openai"           # тот же снимок, что в index.ipynb
GOLDEN_PATH = None                     # None — стандартный fixtures/golden.jsonl; можно указать свой файл
REPEATS = 3                            # сколько раз повторить прогон для честной медианы латентности

## Hit@k, а не общий recall

Попадание засчитывается, если нужный документ нашёлся в топ-k *и* в тексте чанка есть обязательная
фраза (`must` из golden-набора) — так отсекаются случайные совпадения по файлу без реального ответа
внутри. Это Hit@k: доля вопросов с хотя бы одним попаданием, а не полнота по всем релевантным
фрагментам сразу (общий recall при нескольких верных чанках на вопрос).

Замер честный: первый прогон по каждому вопросу — `warmup`, он прогревает модель, соединения и
query-cache и не идёт в статистику латентности; дальше вопросы гоняются `REPEATS` раз, и считаются
медиана (p50) и p95 — устойчивые к редким выбросам оценки, в отличие от среднего. Всё это — внутри
`evaluate()` из `common.py`, здесь только вызов.

## Прогон: golden-вопросы → таблица по четырём ретриверам

Загружаем снимок и golden-набор, считаем эмбеддинги вопросов один раз заранее (а не внутри каждого
поиска — это отдельная от поиска стоимость), затем гоняем `evaluate()` по очереди для dense, BM25,
hybrid RRF и hybrid + rerank из `retrievers.py` дня 2 и печатаем строку таблицы на каждый.

In [ ]:
cfg = load_config(COLLECTION)
golden = golden_for_rows(load_golden(GOLDEN_PATH), load_rows(COLLECTION), require_all=True)
from embed import Embedder
from retrievers import Store
embedder = Embedder(cfg["embedder"], cfg["model"])
t0 = time.perf_counter()
for g in golden:
    embedder.embed_query(g["q"])
print(f"query embeddings (один раз, вне поиска): {time.perf_counter() - t0:.2f} s")
store = Store(COLLECTION, embedder)
print(f"snapshot={cfg['dataset_hash']} questions={len(golden)} repeats={REPEATS}")
print("| retriever | Hit@1 | Hit@3 | Hit@5 | MRR@5 | p50 ms | p95 ms | warmup ms |")
print("|---|---|---|---|---|---|---|---|")
for name, fn in (("dense", store.dense), ("bm25", store.bm25_search), ("hybrid RRF", store.hybrid), ("hybrid + rerank", store.hybrid_rerank)):
    r = evaluate(name, fn, golden, repeats=REPEATS)
    print(f"| {name} | {r['recall@1']:.2f} | {r['recall@3']:.2f} | {r['recall@5']:.2f} | {r['mrr']:.2f} | {r['latency_ms']:.1f} | {r['p95_ms']:.1f} | {r['warmup_ms']:.0f} |")
    if r["misses"]:
        print("Промахи:", " | ".join(r["misses"][:6]))